# Prostate Cancer Biochemical Recurrence Prediction

## Project 12 — Longitudinal Biomarker Trajectory in Oncology

**Course:** I 320D — Data Science for Biomedical Informatics  
**Instructor:** Ammar Darkazanli, Ph.D., MBA  
**Semester:** Spring 2026  
**University of Texas at Austin · School of Information**

---

### 🎯 This Project's Mantra

# "Censored ≠ Cured"

*Translation: A patient who hasn't recurred yet is not the same as a patient who won't recur.*  
*Survival analysis exists precisely to handle this distinction.*

---

## Background

In oncology, biomarkers like **PSA** (prostate), CA-125 (ovarian), and CEA (colorectal) are measured serially over a treatment course. The trajectory — rate of decline after treatment, nadir value, and velocity of rise at relapse — carries more prognostic information than any single measurement.

Modeling biomarker trajectories over treatment cycles allows oncologists to **anticipate response vs. progression weeks before imaging confirmation**. This project introduces mixed-effects modeling, nonlinear trajectory analysis, and survival analysis in a high-stakes clinical context.

---

## What You Will Build

You will predict **biochemical recurrence (BCR)** — a rise in PSA after prostate cancer treatment — in a longitudinal cohort of **600 patients** across four treatment types. You will train:

- A **Logistic Regression** model (classification baseline — Model A: clinical only; Model B: + inflammatory markers)
- A **Linear Mixed-Effects Model** (random intercept + random slope to capture between-patient PSA trajectory variability)
- A **Cox Proportional Hazards** model (survival analysis — the methodological centerpiece)

And you will interpret both models with **SHAP** and **hazard ratio forest plots**.

---

## Dataset Profile

| Property | Value |
|----------|-------|
| File | `prostat_ca_veri_seti_duzeltilmis_v2.csv` |
| Patients | 600 |
| Variables | 31 |
| Outcome | BCR_Durum (True = biochemical recurrence) |
| BCR events | 184 (30.7%) |
| Censored | 416 (69.3%) |
| Mean follow-up | ~59 months |
| Mean time to BCR | ~31 months (among recurred) |

**Treatment types (Tedavi_Tipi):**
- 1 = Surgery (Radical Prostatectomy) — n=254
- 2 = Radiotherapy (RT) — n=182
- 3 = ADT alone — n=71
- 4 = RT + ADT — n=93

**Novel features in this dataset (not in standard public datasets):**  
Inflammatory and nutritional markers — **NLR** (Neutrophil-to-Lymphocyte Ratio), **CALLY Index**, **Albumin**, **CRP** — are measured at baseline. Their incremental predictive value over established clinical features is a primary question of this project.

---

## Instructions

1. Complete each cell marked with `# TODO:`
2. Run your code to verify it works before moving on
3. Answer the **Challenge Questions** in the markdown cells below each step
4. Use the hints provided if you get stuck
5. Ask questions during class if needed!


---
# STEP 0 — Environment Setup & Data Dictionary
---

> ⚠️ **Before you load any data**, complete the data dictionary below.  
> Understanding which columns are **structurally missing** (by treatment design) vs. **randomly missing** (data quality) is the most important analytical skill in this project.
>
> **Key principle from the MD guide:** A student who does not know that `RT_Dozu` is only populated for RT patients will misread 54% missingness as a data quality problem rather than a structural feature.

### Data Dictionary — Complete This Before Running Any Code

| Column | Description | Type | Missingness Hypothesis |
|--------|-------------|------|------------------------|
| Hasta_ID | Patient ID | int | None |
| Yas | Age at diagnosis | int | None |
| Tani_Tarihi | Diagnosis date | date | None |
| PSA_Tani | PSA at diagnosis (ng/mL) | float | None |
| Klinik_Evre | Clinical stage (cT1c–cT3b) | str | None |
| Biyopsi_Gleason | Biopsy Gleason score (e.g. 3+4) | str | None |
| Risk_Grubu | Pre-treatment risk group (1=low, 2=intermediate, 3=high) | int | None |
| Albumin | Serum albumin at baseline (g/dL) | float | None |
| Lenfosit | Lymphocyte count at baseline | int | None |
| CRP | C-reactive protein at baseline | float | None |
| NLR | Neutrophil-to-Lymphocyte Ratio | float | None |
| CALLY_Index | Composite inflammatory-nutritional marker = Albumin × Lenfosit / (NLR × 1000) | float | None |
| Komorbidite_Skor | Comorbidity score | int | None |
| Tedavi_Tipi | Treatment type (1=Surgery, 2=RT, 3=ADT, 4=RT+ADT) | int | None — **encode as categorical, not numeric** |
| Tedavi_Tarihi | Treatment start date | date | None |
| RT_Dozu | Radiotherapy dose (Gy) | float | **Structural** — only RT patients (Tedavi_Tipi 2,4); ~54% missing overall |
| ADT_Tipi | ADT drug type | float | **Structural** — only ADT patients (Tedavi_Tipi 3,4) |
| ADT_Suresi | ADT duration (months) | float | **Structural** — only ADT patients (Tedavi_Tipi 3,4) |
| Patolojik_Evre | Pathological stage (post-surgery) | str | **Structural** — only surgery patients (Tedavi_Tipi 1); ~58% missing overall |
| Cerrahi_Sinir | Surgical margin status | float | **Structural** — only surgery patients |
| Final_Gleason | Post-surgery Gleason (may differ from biopsy) | str | **Structural** — only surgery patients; small random missingness within surgery arm |
| PSA_Nadir | Lowest PSA achieved post-treatment | float | None |
| PSA_Takip_3ay | PSA at 3-month follow-up | float | None |
| PSA_Takip_6ay | PSA at 6-month follow-up | float | None |
| PSA_Takip_12ay | PSA at 12-month follow-up | float | None — ⚠️ post-dates BCR event for some patients |
| BCR_Durum | Biochemical recurrence (True/False) | bool | None |
| BCR_Tarihi | Date of BCR event | str | **Informative** — missing for 416 censored patients (by definition, not data quality) |
| Metastaz_Durum | Metastasis status | int | None |
| Son_Durum | Last known status | int | None |
| Son_Takip_Tarihi | Date of last follow-up | date | None |

> **Tedavi_Tipi encoding risk:** Treating this as numeric (1 < 2 < 3 < 4) implies a false clinical ordering. Always one-hot encode or use the binary treatment flags created in Step 3.


### 0.1 Import Libraries

In [ ]:
# TODO: Import all required libraries
# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Survival analysis
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

# Mixed-effects model (for PSA trajectory modeling — Project 12 docx requirement)
import statsmodels.formula.api as smf

# Machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.calibration import calibration_curve
from scipy.stats import mannwhitneyu

# Interpretability
import shap

# Reproducibility
import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('All libraries imported successfully.')
print('Lifelines version:', __import__('lifelines').__version__)
print('Statsmodels version:', __import__('statsmodels').__version__)


---
# STEP 1 — Data Loading & First Look
---

### 1.1 Load the Dataset

In [ ]:
# TODO: Load the dataset from 'prostat_ca_veri_seti_duzeltilmis_v2.csv'
# Hint: pd.read_csv('prostat_ca_veri_seti_duzeltilmis_v2.csv')
# Then print shape and first 3 rows.

df = pd.read_csv('prostat_ca_veri_seti_duzeltilmis_v2.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

### 1.2 Understand the Outcome Variable

**BCR (Biochemical Recurrence)** is a **time-to-event outcome** with censoring — defined differently by treatment type:
- **After surgery (RP):** PSA ≥ 0.2 ng/mL on two consecutive measurements
- **After radiotherapy (Phoenix criterion):** PSA rise ≥ 2 ng/mL above nadir

The `BCR_Durum` column **already encodes these treatment-specific criteria**. The label is not a simple uniform threshold.

**Related outcome (docx Project 12):** PSA50 response = PSA decline of ≥50% from baseline. All 600 patients in this cohort achieved nadir_depth < 0.5 (PSA50 response), but the *depth* of suppression varies prognostically.

> ⚠️ **Core pitfall:** Treating `BCR_Durum = False` as "no BCR" is wrong for survival modeling. These 416 patients were simply followed until `Son_Takip_Tarihi` — some will eventually recur. This is **censoring**, not proof of cure.


In [ ]:
# TODO: Print the BCR rate overall and by treatment type (Tedavi_Tipi)
# Hint: df.groupby('Tedavi_Tipi')['BCR_Durum'].mean()

print('=== BCR_Durum Distribution ===')
print(df['BCR_Durum'].value_counts())
print(f'\nOverall BCR rate: {df["BCR_Durum"].mean()*100:.1f}%')

print('\n=== BCR Rate by Treatment Type ===')
treatment_labels = {1: 'Surgery', 2: 'RT', 3: 'ADT', 4: 'RT+ADT'}
bcr_by_tx = df.groupby('Tedavi_Tipi')['BCR_Durum'].agg(['mean', 'sum', 'count'])
bcr_by_tx.index = bcr_by_tx.index.map(treatment_labels)
bcr_by_tx.columns = ['BCR Rate', 'BCR Events', 'N Patients']
bcr_by_tx['BCR Rate'] = bcr_by_tx['BCR Rate'].map('{:.1%}'.format)
print(bcr_by_tx)

### 1.3 Compute Survival Duration

For survival modeling, every patient needs a **duration** — time from treatment to BCR (for events) or last follow-up (for censored patients). This is not stored directly; you derive it from the date columns.

**Formula:**
- If `BCR_Durum == True`:  duration = (BCR_Tarihi − Tedavi_Tarihi) in months
- If `BCR_Durum == False`: duration = (Son_Takip_Tarihi − Tedavi_Tarihi) in months

In [ ]:
# TODO: Compute 'duration_months' for every patient
# Steps:
#   1. Parse date columns with pd.to_datetime()
#   2. Compute days from Tedavi_Tarihi to BCR_Tarihi (if BCR) or Son_Takip_Tarihi
#   3. Convert days to months (divide by 30.44)
# Hint: Use np.where(condition, value_if_true, value_if_false)

for col in ['Tedavi_Tarihi', 'BCR_Tarihi', 'Son_Takip_Tarihi']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

event_duration   = (df['BCR_Tarihi']       - df['Tedavi_Tarihi']).dt.days
censored_duration = (df['Son_Takip_Tarihi'] - df['Tedavi_Tarihi']).dt.days

df['duration_months'] = np.where(
    df['BCR_Durum'],
    event_duration / 30.44,
    censored_duration / 30.44
)

print('Duration statistics (months):')
print(df['duration_months'].describe().round(1))
print(f'\nPatients with missing duration: {df["duration_months"].isna().sum()}')

### 🤔 Challenge Questions — Step 1

**1a.** `BCR_Tarihi` is missing for 416 patients with `BCR_Durum = False`. For a logistic regression predicting `BCR_Durum`, this is fine. For a survival model, why do you need a duration for these **censored** patients, and what does the duration *mean* clinically for a patient who has not recurred?

*Your answer here:*

---

**1b.** The BCR definition differs by treatment type: ≥ 0.2 ng/mL after surgery vs. nadir + 2 ng/mL after RT. If `BCR_Durum` already applies these criteria, what implication does this have for building a **single pooled model**? Does your model learn a unified risk signal, or is it fitting treatment-specific thresholds?

*Your answer here:*

---

**1c.** The docx asks: *"If you observe only 3 PSA measurements per patient, how precisely can you estimate an individual's trajectory slope? What does this mean for prediction uncertainty?"* Apply this question to your dataset — how many PSA timepoints does each patient have, and what is the implication for PSADT reliability?

*Your answer here:*


---
# STEP 2 — Exploratory Data Analysis (EDA)
---

### 2.1 Class Balance

In [ ]:
# TODO: Create a bar chart of BCR_Durum counts (True vs False)
# and a grouped bar chart showing BCR rate by treatment type

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overall BCR distribution
counts = df['BCR_Durum'].value_counts()
axes[0].bar(['No BCR (Censored)', 'BCR (Event)'], counts.values,
            color=['#4C9BE8', '#E84C4C'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Overall BCR Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)

# BCR rate by treatment type
tx_names = {1: 'Surgery\n(n=254)', 2: 'RT\n(n=182)', 3: 'ADT\n(n=71)', 4: 'RT+ADT\n(n=93)'}
bcr_rates = df.groupby('Tedavi_Tipi')['BCR_Durum'].mean()
bars = axes[1].bar([tx_names[k] for k in bcr_rates.index], bcr_rates.values * 100,
                   color=['#5DA0D0', '#E8A84C', '#8BC94C', '#C44CDB'], edgecolor='white')
axes[1].axhline(df['BCR_Durum'].mean() * 100, color='red', linestyle='--', alpha=0.7, label='Overall mean')
axes[1].set_title('BCR Rate by Treatment Type', fontweight='bold')
axes[1].set_ylabel('BCR Rate (%)')
axes[1].legend()
for bar, rate in zip(bars, bcr_rates.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{rate*100:.1f}%', ha='center', fontsize=10)

plt.suptitle('Outcome Distribution — Prostate Cancer BCR Cohort', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Structural Missingness Heatmap

The most important EDA insight in this dataset: **missingness is not random — it encodes which treatment a patient received.**

- `Patolojik_Evre`, `Cerrahi_Sinir`, `Final_Gleason` → only surgery patients (~58% missing overall)
- `RT_Dozu` → only RT patients (Tedavi_Tipi 2, 4) (~54% missing overall)
- `ADT_Tipi`, `ADT_Suresi` → only ADT patients (Tedavi_Tipi 3, 4) (~73% missing overall)
- `BCR_Tarihi` → only patients with BCR_Durum = True (69% missing = informative, not structural)

Running a **global** missingness analysis without stratifying by treatment will misdiagnose structural missingness as a data quality failure. **Always stratify the heatmap by Tedavi_Tipi first.**


In [ ]:
# TODO: Create a missingness heatmap stratified by treatment type
# Hint: Compute df.isnull() per Tedavi_Tipi group, show only columns with any missingness

cols_with_missing = df.columns[df.isnull().any()].tolist()

miss_by_tx = df.groupby('Tedavi_Tipi')[cols_with_missing].apply(
    lambda g: g.isnull().mean() * 100
).round(1)
miss_by_tx.index = ['Surgery (1)', 'RT (2)', 'ADT (3)', 'RT+ADT (4)']

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(miss_by_tx, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% Missing'})
ax.set_title('Missingness (%) by Treatment Type — Structural Pattern', fontweight='bold')
ax.set_xlabel('Column')
ax.set_ylabel('Treatment Group')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\nKey insight: Patolojik_Evre / Cerrahi_Sinir / Final_Gleason are ONLY populated for surgery patients.')
print('RT_Dozu is only populated for RT patients. ADT_Tipi / ADT_Suresi only for ADT patients.')

### 2.3 PSA Trajectory by BCR Status

The four follow-up PSA columns encode a longitudinal trajectory across the treatment course:
- **BCR=True patients:** rising PSA after nadir (recurrence signal)
- **BCR=False patients:** continued suppression

**Docx Project 12 requirement:** Create a **spaghetti plot** of individual PSA curves with a population-level model fit overlay. The code below shows the mean ± SE version; the challenge question asks you to extend it to individual trajectories.

> **Informative visit timing (docx):** PSA is measured more frequently when a patient is doing poorly. This means visit density itself may be a signal — patients with more follow-up measurements may be higher risk.


In [ ]:
# TODO: Plot mean PSA trajectory (Nadir → 3m → 6m → 12m) separately for BCR=True vs BCR=False
# Use a line plot with error bands (mean ± SE)

psa_cols = ['PSA_Nadir', 'PSA_Takip_3ay', 'PSA_Takip_6ay', 'PSA_Takip_12ay']
time_labels = ['Nadir', '3 Months', '6 Months', '12 Months']

fig, ax = plt.subplots(figsize=(10, 5))

for bcr_val, label, color in [(True, 'BCR = True (Recurred)', '#E84C4C'),
                               (False, 'BCR = False (Censored)', '#4C9BE8')]:
    subset = df[df['BCR_Durum'] == bcr_val][psa_cols]
    means = subset.mean()
    se    = subset.sem()
    ax.plot(time_labels, means, marker='o', color=color, linewidth=2.5, label=label)
    ax.fill_between(time_labels,
                    means - se, means + se,
                    alpha=0.15, color=color)

ax.set_title('Mean PSA Trajectory: BCR vs. No BCR\n(mean ± SE)', fontweight='bold')
ax.set_xlabel('Follow-up Timepoint')
ax.set_ylabel('PSA (ng/mL)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.4 Inflammatory Marker Distributions

In [ ]:
# TODO: Plot distributions of NLR, CALLY_Index, CRP, and Albumin by BCR status
# Use violin plots or overlapping KDE plots

inflam_markers = ['NLR', 'CALLY_Index', 'CRP', 'Albumin']
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, marker in zip(axes, inflam_markers):
    sns.violinplot(data=df, x='BCR_Durum', y=marker, ax=ax,
                   palette={False: '#4C9BE8', True: '#E84C4C'},
                   inner='quartile', linewidth=1.2)
    ax.set_title(marker, fontweight='bold')
    ax.set_xlabel('BCR Status')
    ax.set_xticklabels(['No BCR', 'BCR'])

plt.suptitle('Inflammatory Markers by BCR Status', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary statistics
print('\nMean values by BCR status:')
print(df.groupby('BCR_Durum')[inflam_markers].mean().round(2))

### 🤔 Challenge Questions — Step 2

**2a.** Plot the distribution of `PSA_Tani` (diagnosis PSA) separately for each of the four treatment groups. Are higher-PSA patients more likely to receive RT+ADT rather than surgery alone? What does this **confounding** imply for a model trained on the pooled cohort?

*Your analysis and answer here:*

---

**2b.** Compare the CALLY Index distribution between BCR=True and BCR=False patients using a Mann-Whitney U test. Now repeat **within each treatment group separately**. Does the association hold, weaken, or reverse? A reversal would indicate **Simpson's paradox** — explain how this can occur.

*Your analysis and answer here:*

---

**2c.** The docx asks: *"PSA is measured at clinic visits, which occur more frequently when a patient is doing poorly. How does 'informative visit timing' bias trajectory estimates?"* In this dataset, do patients with more PSA follow-up timepoints available tend to have higher BCR rates? Test this by comparing BCR rates across patients with complete vs. incomplete follow-up.

*Your analysis and answer here:*

```python
# TODO: Answer Challenge Questions 2a, 2b, and 2c with code
from scipy.stats import mannwhitneyu

# 2a: PSA_Tani distribution by treatment group
fig, ax = plt.subplots(figsize=(10, 5))
# Your code here

# 2b: Mann-Whitney U test for CALLY_Index by BCR status (overall + by treatment)
# Your code here

# 2c: Informative visit timing — BCR rate by PSA follow-up completeness
# Your code here
```


In [ ]:
# TODO: Answer Challenge Questions 2a and 2b with code
from scipy.stats import mannwhitneyu

# 2a: PSA_Tani by treatment group
# Your code here

# 2b: Mann-Whitney U test for CALLY_Index by BCR status
# Your code here

---
# STEP 3 — Missing Data & Clinically-Informed Imputation
---

> ⚠️ **Do not impute treatment-specific columns globally.** Imputing `Patolojik_Evre` for an RT patient using surgery-patient statistics is not just statistically suboptimal — it **fabricates a clinical measurement that was never made**. This is clinically invalid.

### The Correct Approach (MD guide Step 3):

1. **Encode the treatment structure first** → binary flags `had_surgery`, `received_RT`, `received_ADT`
2. **Replace structural missingness with clinically correct indicators** — missing `Patolojik_Evre` for an RT patient is not uncertainty; it *means* the patient did not have surgery
3. **Impute only within clinically appropriate subgroups** and only from training-set statistics
4. **Never use `BCR_Durum` to impute `Final_Gleason`** — this would be direct label leakage

> **Staleness/Leakage Note:** Unlike sepsis where forward-fill can propagate lab values 20+ hours, this dataset has no dense time-series imputation challenges. The key leakage risk is **computing imputation statistics on the full dataset** rather than only the training set.


### 3.1 Encode Treatment Structure as Binary Flags

In [ ]:
# TODO: Create binary treatment indicator columns from Tedavi_Tipi
# had_surgery    = Tedavi_Tipi == 1
# received_RT    = Tedavi_Tipi in (2, 4)
# received_ADT   = Tedavi_Tipi in (3, 4)

df['had_surgery']  = (df['Tedavi_Tipi'] == 1).astype(int)
df['received_RT']  = df['Tedavi_Tipi'].isin([2, 4]).astype(int)
df['received_ADT'] = df['Tedavi_Tipi'].isin([3, 4]).astype(int)

print('Treatment indicator counts:')
print(df[['had_surgery', 'received_RT', 'received_ADT']].sum())
print('\nVerification — surgery patients with Patolojik_Evre:')
print(df.groupby('had_surgery')['Patolojik_Evre'].count())

### 3.2 Within-Group Imputation (Training-Set Only)

In [ ]:
# TODO: For the small number of surgery patients missing Final_Gleason,
# impute using the mode of Final_Gleason among surgery patients.
# For non-surgery patients: leave these columns as NaN (they will be
# dropped from the feature matrix in Step 4).
#
# For continuously distributed columns (CRP, Albumin) with no missing
# values here, no imputation is needed — verify this.

print('Missing values before imputation:')
print(df[['Albumin', 'Lenfosit', 'CRP', 'NLR', 'CALLY_Index']].isnull().sum())

# Impute Final_Gleason for the few missing surgery patients
surgery_mask = df['had_surgery'] == 1
mode_gleason = df.loc[surgery_mask, 'Final_Gleason'].mode()[0]
missing_surgery_gleason = surgery_mask & df['Final_Gleason'].isna()
print(f'\nSurgery patients missing Final_Gleason: {missing_surgery_gleason.sum()}')
print(f'Imputing with mode: {mode_gleason}')
df.loc[missing_surgery_gleason, 'Final_Gleason'] = mode_gleason

print('\n✅ Imputation complete. Core feature columns are clean:')
print(df[['NLR', 'CALLY_Index', 'Albumin', 'CRP', 'PSA_Tani', 'PSA_Nadir']].isnull().sum())

### 🤔 Challenge Questions — Step 3

**3a.** For surgery patients, `Cerrahi_Sinir` (surgical margin status) is fully observed. For all other patients, it is missing. Should you impute this variable for non-surgery patients? If not, how do you handle it in a model that **pools all treatment types**?

*Your answer here:*

---

**3b.** A student proposes imputing missing `Patolojik_Evre` values for RT patients using the **median pathological stage observed in surgery patients**. Describe two distinct clinical errors this introduces — one statistical, one related to treatment confounding.

*Your answer here:*

---

**3c.** Compute the median `Final_Gleason` for surgery patients in the training set, then impute it for the 8 surgery patients who are missing it. Is this imputation clinically defensible? What scenario could cause a surgery patient to have a missing Gleason score?

*Your answer here:*

```python
# TODO: Answer 3c with code — compute and impute Final_Gleason for surgery patients
# Your code here
```


---
# STEP 4 — Feature Engineering
---

No dense time series here — instead we derive **biomarker kinetic features** from the 4 PSA follow-up columns. This is the core distinction from the sepsis project: trajectory features from **sparse longitudinal observations** rather than rolling statistics over dense hourly data.

### New concepts introduced in this step (docx Project 12):

| Feature | Formula | Clinical Meaning |
|---------|---------|-----------------|
| **PSADT** | ln(2) / β where β = slope of log(PSA) vs. time | Time for PSA to double; <10 months = aggressive recurrence |
| **Nadir depth** | PSA_Nadir / PSA_Tani | How deep was the PSA suppression? Deeper = better initial response |
| **PSA velocity** | (PSA_Takip_12ay − PSA_Nadir) / 12 | Rate of PSA rise per month post-nadir |
| **CALLY Index** | Albumin × Lenfosit / (NLR × 1000) | Composite inflammatory-nutritional marker — verify from components |

### Linear Mixed-Effects Model for PSA Trajectory (docx requirement)

The docx Project 12 introduces **random intercept + random slope models** to separate:
- **Fixed effects:** average population-level PSA trajectory across all patients
- **Random effects:** individual patient deviations (some patients have faster/slower trajectories)

Large random-effects variance implies high between-patient heterogeneity — patients in this cohort follow very different PSA trajectories, which is clinically important for personalized prognosis.

> **PSADT doubling time note (docx):** PSADT is a naturally interpretable summary of the log-PSA trajectory. Under the log-linear model, PSADT = ln(2)/β is mathematically equivalent to the inverse of the slope on a log scale. This is a clinically established kinetic metric used in prostate cancer guidelines.


### 4.1 PSA Doubling Time (PSADT)

**Clinical interpretation:** PSADT is the time it takes for PSA to double. Short PSADT (< 10 months) predicts aggressive recurrence. It is derived from a log-linear fit across the 4 follow-up PSA measurements.

$$\log(\text{PSA}) \approx \alpha + \beta \cdot t \quad \Rightarrow \quad \text{PSADT} = \frac{\ln 2}{\beta}$$

where t is measured in months from nadir.

In [ ]:
# TODO: Compute PSADT for each patient
# Steps:
#   1. Create a log-PSA matrix across [Nadir (t=0), 3m, 6m, 12m]
#   2. Fit a linear regression per row: log(PSA) ~ t
#   3. PSADT = ln(2) / slope
#   4. Cap extreme values (e.g., PSADT > 120 months → 120; negative → NaN)

timepoints = np.array([0, 3, 6, 12])

def compute_psadt(row):
    psa_vals = np.array([
        row['PSA_Nadir'], row['PSA_Takip_3ay'],
        row['PSA_Takip_6ay'], row['PSA_Takip_12ay']
    ])
    # Require at least 3 valid, positive PSA values
    valid = psa_vals > 0
    if valid.sum() < 3:
        return np.nan
    t_valid   = timepoints[valid]
    log_psa   = np.log(psa_vals[valid])
    # OLS: slope via closed-form
    t_mean    = t_valid.mean()
    log_mean  = log_psa.mean()
    beta      = np.sum((t_valid - t_mean) * (log_psa - log_mean)) / \
                np.sum((t_valid - t_mean)**2)
    if beta <= 0:
        return np.nan  # Decreasing or flat PSA — no meaningful doubling
    psadt = np.log(2) / beta
    return min(psadt, 120)  # Cap at 120 months

df['PSADT'] = df.apply(compute_psadt, axis=1)

print('PSADT distribution (months):')
print(df['PSADT'].describe().round(1))
print(f'Patients with computable PSADT: {df["PSADT"].notna().sum()} / {len(df)}')
print(f'\nMean PSADT — BCR=True:  {df[df["BCR_Durum"]=="True"]["PSADT"].mean():.1f} months')
print(f'Mean PSADT — BCR=False: {df[df["BCR_Durum"]=="False"]["PSADT"].mean():.1f} months')

### 4.2 Nadir Depth Ratio & PSA Velocity

In [ ]:
# TODO: Compute two additional PSA kinetic features:
#
# nadir_depth = PSA_Nadir / PSA_Tani
#   Interpretation: deeper nadir (smaller ratio) = better initial response
#
# psa_velocity = (PSA_Takip_12ay - PSA_Nadir) / 12
#   Interpretation: rate of PSA rise per month after nadir

df['nadir_depth']   = df['PSA_Nadir'] / df['PSA_Tani']
df['psa_velocity']  = (df['PSA_Takip_12ay'] - df['PSA_Nadir']) / 12

print('Nadir depth (PSA_Nadir / PSA_Tani):')
print(df['nadir_depth'].describe().round(3))
print('\nPSA velocity (ng/mL per month):')
print(df['psa_velocity'].describe().round(4))

### 4.3 Verify the CALLY Index Formula

$$\text{CALLY Index} = \frac{\text{Albumin} \times \text{Lenfosit}}{\text{NLR} \times 1000}$$

In [ ]:
# TODO: Recompute CALLY Index from components and compare to the stored column
# Use np.isclose(a, b, atol=1e-3) to check agreement

df['CALLY_recomputed'] = df['Albumin'] * df['Lenfosit'] / (df['NLR'] * 1000)

agree = np.isclose(df['CALLY_Index'], df['CALLY_recomputed'], atol=1e-3)
print(f'Patients where stored CALLY_Index matches recomputed: {agree.sum()} / {len(df)}')

if not agree.all():
    discrepancies = df[~agree][['CALLY_Index', 'CALLY_recomputed', 'Albumin', 'Lenfosit', 'NLR']]
    print(f'\nDiscrepancies:')
    print(discrepancies.head())
else:
    print('✅ Formula verified — all values match.')

df.drop(columns=['CALLY_recomputed'], inplace=True)

### 4.4 Encode Categorical Features

In [ ]:
# TODO: Encode categorical predictors for modeling
#
# Klinik_Evre: ordinal — cT1c < cT2a < cT2b < cT2c < cT3a < cT3b
# Biyopsi_Gleason: extract Gleason sum (e.g. '3+4' -> 7)
# Risk_Grubu: already numeric (1, 2, 3) — keep as-is
# Tedavi_Tipi: one-hot encode (already have binary flags from Step 3)

# Ordinal Klinik_Evre
evre_order = {'cT1c': 1, 'cT2a': 2, 'cT2b': 3, 'cT2c': 4, 'cT3a': 5, 'cT3b': 6}
df['Klinik_Evre_num'] = df['Klinik_Evre'].map(evre_order)

# Gleason sum from string e.g. '3+4' -> 7
df['Gleason_sum'] = df['Biyopsi_Gleason'].apply(
    lambda x: sum(int(g) for g in str(x).split('+')) if pd.notna(x) else np.nan
)

print('Klinik_Evre encoding:')
print(df[['Klinik_Evre', 'Klinik_Evre_num']].drop_duplicates().sort_values('Klinik_Evre_num'))
print('\nGleason sum distribution:')
print(df['Gleason_sum'].value_counts().sort_index())

### 🤔 Challenge Questions — Step 4

**4a.** For a patient with PSA_Nadir = 0.04, PSA_Takip_3ay = 0.04, PSA_Takip_6ay = 0.06, PSA_Takip_12ay = 0.12, compute the PSADT **manually** using the formula above. Is this patient at high or low risk of BCR by the clinical threshold (PSADT < 10 months = high risk)?

*Your calculation here:*

---

**4b.** Why is treating `Tedavi_Tipi` as a numeric feature (1 < 2 < 3 < 4) incorrect? What ordering does this imply, and is that ordering clinically meaningful? What encoding is correct for logistic regression vs. a tree-based model?

*Your answer here:*

---

**4c.** The docx asks: *"PSA doubling time is a commonly used clinical metric. Is this the same as the slope of a log-PSA trajectory? Under what model is doubling time a natural summary statistic?"* Answer in 2–3 sentences, then verify by checking whether log-linear PSADT computed from your function matches a direct ratio-based estimate for a sample of 5 patients.

*Your answer and code here:*

```python
# TODO: Verify PSADT computation for 5 sample patients
# Compare your compute_psadt() output vs. direct ln(2)/slope estimate
# Your code here
```


---
# STEP 5 — Train / Test Split (Patient-Level)
---

> ⚠️ **Temporal leakage risk (MD guide Step 5):** For patients whose BCR occurred *before* the 12-month follow-up window, `PSA_Takip_12ay` reflects a measurement taken **after** the event. Including it as a predictor is leakage. We use only `PSA_Nadir` and the 3-month follow-up as safe predictors in the baseline model.

### Landmark Analysis (docx Project 12)

The docx introduces **landmark analysis** as the correct approach for wide-format longitudinal data:
- Fix a landmark time (e.g., 12 months post-treatment)
- **Include only patients who have not had BCR by the landmark**
- Predict future BCR using only features available at the landmark date

This prevents the most common form of temporal leakage in oncology data: using post-event measurements to predict the event.

### Stratification Key

Unlike the sepsis project where the split error is row-level vs. patient-level, here the risk is subtler: if you stratify only by BCR_Durum and ignore Tedavi_Tipi, treatment imbalance in the test set can confound evaluation. **Always stratify by both.**


In [ ]:
# TODO: Build the feature matrix X and label vector y
# Include: Yas, PSA_Tani, Klinik_Evre_num, Gleason_sum, Risk_Grubu,
#          Albumin, CRP, NLR, CALLY_Index, Komorbidite_Skor,
#          PSA_Nadir, nadir_depth, PSADT, psa_velocity,
#          had_surgery, received_RT, received_ADT
#
# Drop rows with NaN in any feature (or impute PSADT median)

FEATURES = [
    'Yas', 'PSA_Tani', 'Klinik_Evre_num', 'Gleason_sum', 'Risk_Grubu',
    'Albumin', 'CRP', 'NLR', 'CALLY_Index', 'Komorbidite_Skor',
    'PSA_Nadir', 'nadir_depth', 'PSADT', 'psa_velocity',
    'had_surgery', 'received_RT', 'received_ADT'
]

# Impute PSADT with column median before dropping
df['PSADT'] = df['PSADT'].fillna(df['PSADT'].median())
df['psa_velocity'] = df['psa_velocity'].fillna(df['psa_velocity'].median())

X = df[FEATURES].copy()
y = df['BCR_Durum'].astype(int)

# Drop any remaining NaN rows
valid_rows = X.notna().all(axis=1)
X, y = X[valid_rows], y[valid_rows]
df_model = df[valid_rows].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'BCR rate in model dataset: {y.mean()*100:.1f}%')
print(f'\nFeatures used:')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# TODO: Stratified 80/20 train/test split
# Stratify by BOTH BCR_Durum and Tedavi_Tipi simultaneously
# Hint: create a combined stratum label: y.astype(str) + '_' + df_model['Tedavi_Tipi'].astype(str)

stratify_key = y.astype(str) + '_' + df_model['Tedavi_Tipi'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify_key
)

# Corresponding full-row splits for survival modeling
train_idx = X_train.index
test_idx  = X_test.index
df_train  = df_model.loc[train_idx]
df_test   = df_model.loc[test_idx]

print(f'Train: {len(X_train)} patients | BCR rate: {y_train.mean()*100:.1f}%')
print(f'Test:  {len(X_test)}  patients | BCR rate: {y_test.mean()*100:.1f}%')
print('\nTreatment distribution in train:')
print(df_train['Tedavi_Tipi'].value_counts().sort_index())
print('\nTreatment distribution in test:')
print(df_test['Tedavi_Tipi'].value_counts().sort_index())

In [ ]:
# Landmark leakage check — identify patients where BCR occurred before 12-month follow-up
# For these patients, PSA_Takip_12ay was measured AFTER the BCR event

df['Tedavi_Tarihi_dt'] = pd.to_datetime(df['Tedavi_Tarihi'], errors='coerce')
df['BCR_Tarihi_dt']    = pd.to_datetime(df['BCR_Tarihi'],    errors='coerce')

# Landmark = 12 months (365 days) after treatment
df['landmark_date'] = df['Tedavi_Tarihi_dt'] + pd.Timedelta(days=365)

# Flag BCR patients whose event occurred before the 12-month landmark
bcr_before_landmark = (
    df['BCR_Durum'] == True
) & (
    df['BCR_Tarihi_dt'] < df['landmark_date']
)

n_leaked = bcr_before_landmark.sum()
print(f'Patients with BCR before 12-month landmark: {n_leaked} / {df["BCR_Durum"].sum()} BCR events')
print(f'  → PSA_Takip_12ay for these patients was measured AFTER their BCR event.')
print(f'  → We address this by EXCLUDING PSA_Takip_12ay and PSA_Takip_6ay from the feature set.')
print(f'  → Only PSA_Nadir and PSA_Takip_3ay are used as trajectory predictors (safe at any BCR timing).')

df['bcr_before_landmark'] = bcr_before_landmark.astype(int)
print(f'\nBCR-before-landmark flag added to df.')


### 🤔 Challenge Questions — Step 5

**5a.** The leakage check above identified patients where `BCR_Tarihi` falls before the 12-month follow-up window. How does our feature selection (using only PSA_Nadir and PSA_Takip_3ay) address this risk? Could PSA_Takip_3ay also be leaked for any patient subset?

*Your answer here:*

---

**5b.** Rerun the train/test split stratifying **only by `BCR_Durum`** (ignoring `Tedavi_Tipi`). Compare the treatment distribution in the resulting test set vs. the properly stratified split. How large is the treatment imbalance, and what would it mean for model evaluation?

*Your analysis here:*

```python
# TODO: Demonstrate the impact of ignoring treatment stratification
# Compare treatment distributions between the two split strategies
# Your code here
```

---

**5c.** The docx proposes a **cross-treatment generalization test**: train on surgery + RT patients, test on ADT patients. What assumption does this test? Implement it and report the AUROC. Does the model generalize?

*Your analysis and code here:*


---
# STEP 6 — Baseline Model: Logistic Regression
---

### 6.1 Train Full Model and Clinical-Only Ablation

In [ ]:
# TODO: Train two logistic regression models:
#   Model A: clinical features only (Yas, PSA_Tani, Klinik_Evre_num, Gleason_sum, Risk_Grubu)
#   Model B: clinical + inflammatory markers (add NLR, CALLY_Index, CRP, Albumin)
# Both: class_weight='balanced', StandardScaler, C=1.0

CLINICAL_FEATURES = ['Yas', 'PSA_Tani', 'Klinik_Evre_num', 'Gleason_sum', 'Risk_Grubu',
                     'PSA_Nadir', 'nadir_depth', 'PSADT', 'psa_velocity',
                     'had_surgery', 'received_RT', 'received_ADT']
INFLAM_FEATURES   = ['NLR', 'CALLY_Index', 'CRP', 'Albumin']

scaler_A = StandardScaler()
scaler_B = StandardScaler()

X_train_A = scaler_A.fit_transform(X_train[CLINICAL_FEATURES])
X_test_A  = scaler_A.transform(X_test[CLINICAL_FEATURES])

X_train_B = scaler_B.fit_transform(X_train[CLINICAL_FEATURES + INFLAM_FEATURES])
X_test_B  = scaler_B.transform(X_test[CLINICAL_FEATURES + INFLAM_FEATURES])

lr_A = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr_B = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)

lr_A.fit(X_train_A, y_train)
lr_B.fit(X_train_B, y_train)

prob_A = lr_A.predict_proba(X_test_A)[:, 1]
prob_B = lr_B.predict_proba(X_test_B)[:, 1]

auc_A  = roc_auc_score(y_test, prob_A)
auprc_A = average_precision_score(y_test, prob_A)
auc_B  = roc_auc_score(y_test, prob_B)
auprc_B = average_precision_score(y_test, prob_B)

print('=== Model Comparison ===')
print(f'Model A (Clinical only):           AUROC = {auc_A:.3f}  |  AUPRC = {auprc_A:.3f}')
print(f'Model B (Clinical + Inflammatory): AUROC = {auc_B:.3f}  |  AUPRC = {auprc_B:.3f}')
print(f'\nIncremental AUROC from inflammatory markers: {auc_B - auc_A:+.3f}')

### 6.2 Clinical Comparator: Risk_Grubu Alone

In [ ]:
# TODO: Evaluate Risk_Grubu alone as a predictor (clinician's pre-treatment risk group)
# Scale it to [0,1] and compute AUROC
# Then plot ROC curves for Model A, Model B, and Risk_Grubu together

risk_score = X_test['Risk_Grubu'] / 3.0  # Normalize 1-3 to [0.33, 1.0]
auc_risk = roc_auc_score(y_test, risk_score)

fig, ax = plt.subplots(figsize=(8, 7))

for prob, label, color in [
    (prob_B,     f'Model B: Clinical + Inflammatory (AUC={auc_B:.3f})', '#2196F3'),
    (prob_A,     f'Model A: Clinical Only (AUC={auc_A:.3f})',           '#4CAF50'),
    (risk_score, f'Risk_Grubu alone (AUC={auc_risk:.3f})',              '#FF9800'),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, label=label, linewidth=2)

ax.plot([0,1], [0,1], 'k--', alpha=0.4, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — BCR Prediction Models', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Treatment-Stratified Evaluation

In [ ]:
# TODO: Compute AUROC for Model B separately within each treatment group
# Hint: iterate over Tedavi_Tipi values in the test set

tx_labels = {1: 'Surgery', 2: 'RT', 3: 'ADT', 4: 'RT+ADT'}
print('=== Treatment-Stratified AUROC (Model B) ===')
for tx_val, tx_name in tx_labels.items():
    mask = df_test['Tedavi_Tipi'] == tx_val
    if mask.sum() < 10:
        print(f'{tx_name}: Too few patients (n={mask.sum()}) for reliable AUROC')
        continue
    y_sub = y_test[mask]
    p_sub = prob_B[mask.values]
    if y_sub.nunique() < 2:
        print(f'{tx_name}: No positive cases in test set')
        continue
    auc_sub = roc_auc_score(y_sub, p_sub)
    print(f'{tx_name:10s} (n={mask.sum():3d}): AUROC = {auc_sub:.3f}')

print(f'\nPooled overall: AUROC = {auc_B:.3f}')

### 🤔 Challenge Questions — Step 6

**6a.** If you reported accuracy instead of AUROC, what accuracy would you get by always predicting "no BCR"? Why is accuracy misleading here?

*Your answer here:*

---

**6b.** Does Model B significantly outperform Model A? Use DeLong's test or bootstrap confidence intervals on the AUROC difference. What would you conclude about the incremental value of inflammatory markers?

*Your analysis here:*

---
# STEP 6b — Linear Mixed-Effects Model for PSA Trajectory
---

> This step addresses the **docx Project 12** requirement: *"Model PSA trajectories across treatment cycles using linear mixed-effects models."*

A linear mixed-effects model (LME) separates:
- **Fixed effects (β):** the average population-level trajectory — how PSA changes over time for the average patient
- **Random effects (u_i):** patient-specific deviations — some patients have faster/slower trajectories than the population average

**Why this matters over a simple mean:** Two patients can have identical population-average predicted PSA but very different individual trajectories. Large random-effect variance reveals **heterogeneity** that a single population model cannot capture.

$$\log(\text{PSA}_{it}) = (\beta_0 + u_{0i}) + (\beta_1 + u_{1i}) \cdot t + \epsilon_{it}$$

Where:
- $\beta_0$ = population intercept (average log-PSA at nadir)
- $u_{0i}$ = patient-specific intercept deviation
- $\beta_1$ = population slope (average log-PSA change per month)
- $u_{1i}$ = patient-specific slope deviation (faster/slower risers)


In [ ]:
# Mixed-Effects Model: Random intercept + random slope for log-PSA trajectory
# Requires: statsmodels (imported in Step 0)
#
# Reshape PSA data to long format: one row per (patient × timepoint)
# Timepoints: Nadir (t=0), 3m (t=3), 6m (t=6), 12m (t=12)

psa_wide = df[['Hasta_ID', 'BCR_Durum', 'Tedavi_Tipi',
               'PSA_Nadir', 'PSA_Takip_3ay', 'PSA_Takip_6ay', 'PSA_Takip_12ay']].copy()

psa_long = psa_wide.melt(
    id_vars=['Hasta_ID', 'BCR_Durum', 'Tedavi_Tipi'],
    value_vars=['PSA_Nadir', 'PSA_Takip_3ay', 'PSA_Takip_6ay', 'PSA_Takip_12ay'],
    var_name='timepoint', value_name='PSA'
)

# Map timepoint names to numeric months
timepoint_map = {'PSA_Nadir': 0, 'PSA_Takip_3ay': 3, 'PSA_Takip_6ay': 6, 'PSA_Takip_12ay': 12}
psa_long['months'] = psa_long['timepoint'].map(timepoint_map)

# Log-transform PSA (add small epsilon to handle near-zero values)
psa_long = psa_long[psa_long['PSA'] > 0].copy()
psa_long['log_PSA'] = np.log(psa_long['PSA'])

print(f'Long-format PSA dataset: {psa_long.shape[0]} rows ({psa_long["Hasta_ID"].nunique()} patients × up to 4 timepoints)')
print(psa_long.head(8))

# Fit mixed-effects model: random intercept + random slope per patient
# Formula: log_PSA ~ months + BCR_Durum + (random intercept + slope by Hasta_ID)
try:
    lme = smf.mixedlm(
        "log_PSA ~ months + C(BCR_Durum)",
        data=psa_long,
        groups=psa_long["Hasta_ID"],
        exog_re=psa_long[["months"]]  # Random slope on time
    )
    lme_result = lme.fit(method='lbfgs', maxiter=500)
    print("\n=== Mixed-Effects Model Summary ===")
    print(lme_result.summary())

    # Extract variance components
    print("\n=== Variance Components ===")
    print(f"Fixed effect (average slope): {lme_result.params['months']:.4f} log-PSA units/month")
    print(f"  → Average PSA change per month: {np.exp(lme_result.params['months']):.4f}x")
    print(f"BCR_Durum[T.True] fixed effect: {lme_result.params.get('C(BCR_Durum)[T.True]', 'N/A')}")

except Exception as e:
    print(f"Mixed-effects fitting note: {e}")
    print("\nFallback: Fitting simpler random-intercept-only model...")
    lme_simple = smf.mixedlm("log_PSA ~ months + C(BCR_Durum)", data=psa_long, groups=psa_long["Hasta_ID"])
    lme_result = lme_simple.fit()
    print(lme_result.summary())


In [ ]:
# Spaghetti plot: Individual PSA trajectories + population-level model fit
# (Docx Project 12 final outcome: "Spaghetti plot of individual PSA curves with population-level model fit overlay")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
timepoints_plot = [0, 3, 6, 12]
time_labels_plot = ['Nadir', '3m', '6m', '12m']
psa_cols_plot = ['PSA_Nadir', 'PSA_Takip_3ay', 'PSA_Takip_6ay', 'PSA_Takip_12ay']

for ax_idx, (bcr_val, bcr_label, color, n_sample) in enumerate([
    (True,  'BCR = True (Recurred)',   '#E84C4C', 40),
    (False, 'BCR = False (Censored)',  '#4C9BE8', 40)
]):
    ax = axes[ax_idx]
    subset = df[df['BCR_Durum'] == bcr_val].sample(min(n_sample, df['BCR_Durum'].eq(bcr_val).sum()),
                                                    random_state=RANDOM_STATE)

    # Individual trajectories (spaghetti)
    for _, row in subset.iterrows():
        vals = [row[c] for c in psa_cols_plot]
        if all(v > 0 for v in vals):
            ax.plot(timepoints_plot, vals, color=color, alpha=0.15, linewidth=0.8)

    # Population mean ± SE overlay
    pop_means = df[df['BCR_Durum'] == bcr_val][psa_cols_plot].mean()
    pop_se    = df[df['BCR_Durum'] == bcr_val][psa_cols_plot].sem()
    ax.plot(timepoints_plot, pop_means, color=color, linewidth=3,
            label=f'Population mean (n={df["BCR_Durum"].eq(bcr_val).sum()})', zorder=5)
    ax.fill_between(timepoints_plot, pop_means - pop_se, pop_means + pop_se,
                    alpha=0.3, color=color)

    ax.set_title(f'PSA Trajectories: {bcr_label}', fontweight='bold')
    ax.set_xlabel('Months from Treatment')
    ax.set_ylabel('PSA (ng/mL)')
    ax.set_xticks(timepoints_plot)
    ax.set_xticklabels(time_labels_plot)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Individual PSA Trajectories with Population-Level Overlay\n(Spaghetti Plot — Project 12 Requirement)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey observation from the mixed-effects model:")
print("  - BCR=True patients show rising PSA after nadir (positive slope)")
print("  - BCR=False patients show flat or declining PSA (near-zero or negative slope)")
print("  - Large variance in individual trajectories confirms need for random effects")


---
# STEP 7 — Survival Model: Cox Proportional Hazards
---

The Cox model is the **principled solution** when your outcome has **censoring**. Here is why it matters:

| Situation | Logistic Regression | Cox Model |
|-----------|--------------------|-----------| 
| BCR_Durum = True (recurred) | Correctly labeled as positive | Event at known duration |
| BCR_Durum = False (censored) | **Treated as "no BCR" — wrong for short follow-up** | Correctly treated as censored — patient was event-free *until* Son_Takip_Tarihi |
| Patient with 6-month follow-up, no BCR | Counted as negative | Contributes only 6 months of risk information |

**Inputs:**
- `duration_months`: time from treatment to BCR or last follow-up
- `BCR_Durum`: 1 = event, 0 = censored
- Feature columns (same as logistic regression)

**New concept (MD guide Step 7):** Verify the **proportional hazards assumption** using Schoenfeld residuals. If NLR or PSADT have time-varying effects (e.g., NLR matters more in the first 24 months than later), the PH assumption is violated and a time-stratified Cox or flexible parametric model is needed.

**Time-varying extension (docx Project 12):** PSA_Takip_3ay, 6ay, 12ay can be incorporated as time-varying covariates in an extended Cox model — the most natural extension once the baseline model is working.


### 7.1 Kaplan-Meier Curves by PSADT Tertile

In [ ]:
# TODO: Plot Kaplan-Meier BCR-free survival curves stratified by PSADT tertile
# Use lifelines.KaplanMeierFitter
# Perform log-rank test between the three groups

df_km = df_train.copy()
df_km['PSADT_tertile'] = pd.qcut(df_km['PSADT'], q=3, labels=['Low PSADT\n(Fast)', 'Medium PSADT', 'High PSADT\n(Slow)'])

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E84C4C', '#FF9800', '#4CAF50']

for (label, group), color in zip(df_km.groupby('PSADT_tertile'), colors):
    kmf = KaplanMeierFitter()
    kmf.fit(group['duration_months'], event_observed=group['BCR_Durum'],
            label=f'{label} (n={len(group)})')
    kmf.plot_survival_function(ax=ax, color=color, ci_show=True)

# Log-rank test (low vs high)
low_grp  = df_km[df_km['PSADT_tertile'] == 'Low PSADT\n(Fast)']
high_grp = df_km[df_km['PSADT_tertile'] == 'High PSADT\n(Slow)']
lr_result = logrank_test(low_grp['duration_months'], high_grp['duration_months'],
                          event_observed_A=low_grp['BCR_Durum'],
                          event_observed_B=high_grp['BCR_Durum'])

ax.set_title(f'Kaplan-Meier BCR-Free Survival by PSADT Tertile\nLog-rank p (Low vs High) = {lr_result.p_value:.4f}',
             fontweight='bold')
ax.set_xlabel('Time (Months from Treatment)')
ax.set_ylabel('BCR-Free Survival Probability')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.2 Fit the Cox Proportional Hazards Model

In [ ]:
# TODO: Build the survival dataframe for lifelines CoxPHFitter
# Required format: DataFrame with columns for duration, event, and all features
# Fit using CoxPHFitter(penalizer=0.1)  -- L2 regularization for correlated inflammatory markers

COX_FEATURES = CLINICAL_FEATURES + INFLAM_FEATURES

survival_train = df_train[COX_FEATURES + ['duration_months', 'BCR_Durum']].copy()
survival_test  = df_test[COX_FEATURES  + ['duration_months', 'BCR_Durum']].copy()

# Scale continuous features for numerical stability
scaler_cox = StandardScaler()
survival_train[COX_FEATURES] = scaler_cox.fit_transform(survival_train[COX_FEATURES])
survival_test[COX_FEATURES]  = scaler_cox.transform(survival_test[COX_FEATURES])

cox = CoxPHFitter(penalizer=0.1)
cox.fit(survival_train, duration_col='duration_months', event_col='BCR_Durum')

cox.print_summary(decimals=3)

# Concordance index on test set
c_index_test = cox.score(survival_test, scoring_method='concordance_index')
print(f'\nTest set C-index: {c_index_test:.3f}')

### 7.3 Individual Survival Curve Predictions

In [ ]:
# TODO: Plot predicted BCR-free survival curves for 3 representative test patients:
#   - Patient at the 10th percentile of predicted risk (low risk)
#   - Patient at the 50th percentile (median risk)
#   - Patient at the 90th percentile (high risk)

risk_scores = cox.predict_partial_hazard(survival_test)
percentiles  = [10, 50, 90]
labels_map   = {10: 'Low Risk (10th percentile)', 50: 'Median Risk', 90: 'High Risk (90th percentile)'}
colors_map   = {10: '#4CAF50', 50: '#FF9800', 90: '#E84C4C'}

fig, ax = plt.subplots(figsize=(10, 6))

for pct in percentiles:
    threshold = np.percentile(risk_scores, pct)
    idx = (risk_scores - threshold).abs().idxmin()
    sf = cox.predict_survival_function(survival_test.loc[[idx]])
    ax.plot(sf.index, sf.values.flatten(),
            label=labels_map[pct], color=colors_map[pct], linewidth=2.5)

ax.set_title('Predicted BCR-Free Survival Curves by Risk Percentile', fontweight='bold')
ax.set_xlabel('Time (Months)')
ax.set_ylabel('Probability of Remaining BCR-Free')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🤔 Challenge Questions — Step 7

**7a.** Fit both a logistic regression (ignoring censoring) and a Cox model on the same training set. Compare their predicted risk rankings using Spearman correlation. For which patients do the two models **disagree most**? What do these patients have in common (hint: think about follow-up duration)?

*Your analysis here:*

```python
# TODO: Spearman correlation between LR and Cox risk rankings
# Identify high-disagreement patients and their follow-up characteristics
from scipy.stats import spearmanr
# Your code here
```

---

**7b.** Check the proportional hazards assumption for NLR using `cox.check_assumptions(survival_train)`. Does NLR have a time-varying effect? What would this mean clinically — does systemic inflammation matter more in the early post-treatment period than at 5-year follow-up?

*Your analysis here:*

---

**7c.** Plot Kaplan-Meier survival curves for each **tertile of PSADT** (as in cell 7.1). Does shorter PSADT correspond to faster BCR, as clinical intuition predicts? Does the separation between curves suggest PSADT is a stronger predictor than `Risk_Grubu` alone?

*Your analysis here:*


---
# STEP 8 — Evaluation: Beyond AUROC
---

### 8.1 Summary Metrics: Classification + Survival

In [ ]:
# TODO: Compile a summary table comparing:
#   - Risk_Grubu alone
#   - Logistic Regression A (clinical)
#   - Logistic Regression B (clinical + inflammatory)
#   - Cox PH model (C-index)

print('='*65)
print(f'{"Model":<35}  {"AUROC":>8}  {"AUPRC":>8}  {"C-index":>8}')
print('-'*65)
print(f'{"Risk_Grubu (clinical heuristic)":<35}  {auc_risk:>8.3f}  {average_precision_score(y_test, risk_score):>8.3f}  {"N/A":>8}')
print(f'{"LR Model A (clinical only)":<35}  {auc_A:>8.3f}  {auprc_A:>8.3f}  {"N/A":>8}')
print(f'{"LR Model B (+ inflammatory)":<35}  {auc_B:>8.3f}  {auprc_B:>8.3f}  {"N/A":>8}')
print(f'{"Cox PH Model":<35}  {"N/A":>8}  {"N/A":>8}  {c_index_test:>8.3f}')
print('='*65)
print('\nNote: AUROC and C-index are mathematically equivalent for uncensored data.')
print('C-index generalizes AUROC to censored survival outcomes.')

### 8.2 Calibration Plot

In [ ]:
# TODO: Plot a calibration curve for Model B (LR)
# Bin test patients by predicted probability (deciles)
# Plot mean predicted probability vs. observed BCR rate per bin

from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 7))

frac_pos, mean_pred = calibration_curve(y_test, prob_B, n_bins=8, strategy='quantile')
ax.plot(mean_pred, frac_pos, 's-', color='#2196F3', linewidth=2, markersize=8,
        label='Model B (LR + Inflammatory)')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')

ax.set_title('Calibration Plot: LR Model B', fontweight='bold')
ax.set_xlabel('Mean Predicted BCR Probability')
ax.set_ylabel('Observed BCR Fraction')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🤔 Challenge Questions — Step 8

**8a.** Compute the Cox C-index separately for **surgery patients** and **RT+ADT patients** in the test set. If the C-index differs substantially (> 0.10), what does this tell you about which patient population drives the model's pooled performance? What could explain the discrepancy?

*Your analysis here:*

```python
# TODO: Subgroup C-index for surgery vs. RT+ADT patients
# Your code here
```

---

**8b.** In your calibration plot, if patients predicted at 70% BCR probability actually recur at only 50%, what clinical decision would be made incorrectly? Who bears the cost of miscalibration in this prostate cancer setting vs. an ICU setting?

*Your answer here:*

---

**8c.** The docx Project 12 asks: *"Your model predicts recurrence 6 months earlier than standard clinical detection. What clinical evidence would be required before acting on this earlier signal?"* Answer this in 3–4 sentences, referencing the concepts of calibration and prospective validation.

*Your answer here:*


---
# STEP 9 — Model Interpretation
---

### 9.1 SHAP Beeswarm Plot — Logistic Regression Model B

In [ ]:
# TODO: Compute SHAP values for Model B using shap.LinearExplainer
# Plot a beeswarm summary plot showing which features drive high BCR probability

all_features = CLINICAL_FEATURES + INFLAM_FEATURES
X_test_B_df  = pd.DataFrame(X_test_B, columns=all_features)
X_train_B_df = pd.DataFrame(X_train_B, columns=all_features)

explainer = shap.LinearExplainer(lr_B, X_train_B_df)
shap_values = explainer.shap_values(X_test_B_df)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_B_df, plot_type='beeswarm', show=False,
                  max_display=16)
plt.title('SHAP Feature Importance — LR Model B (BCR Prediction)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 9.2 Local Explanation — Waterfall Plot for a High-Risk Patient

In [ ]:
# TODO: Select the test patient with the highest predicted BCR probability from Model B
# Generate a waterfall plot showing how each feature pushes their risk up or down

high_risk_idx_local = np.argmax(prob_B)
print(f'Highest-risk test patient (local index {high_risk_idx_local}):')
print(f'  Predicted BCR probability: {prob_B[high_risk_idx_local]:.1%}')
print(f'  Actual BCR status: {y_test.iloc[high_risk_idx_local]}')

shap_explanation = shap.Explanation(
    values=shap_values[high_risk_idx_local],
    base_values=explainer.expected_value,
    data=X_test_B_df.iloc[high_risk_idx_local],
    feature_names=all_features
)

plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_explanation, show=False, max_display=15)
plt.title('SHAP Waterfall — Highest-Risk Patient', fontweight='bold')
plt.tight_layout()
plt.show()

### 9.3 Cox Hazard Ratio Forest Plot

In [ ]:
# TODO: Extract hazard ratios and 95% CIs from the Cox model
# Plot a forest plot showing HR for all features
# Color features by whether their 95% CI crosses HR=1.0 (null effect)

cox_summary = cox.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
cox_summary.columns = ['HR', 'CI_low', 'CI_high', 'p_val']
cox_summary = cox_summary.sort_values('HR', ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))

y_positions = range(len(cox_summary))
colors_hr = ['#E84C4C' if row['CI_low'] > 1 else
              '#4CAF50' if row['CI_high'] < 1 else
              '#9E9E9E'
              for _, row in cox_summary.iterrows()]

ax.scatter(cox_summary['HR'], y_positions, color=colors_hr, zorder=3, s=80)
for i, (_, row) in enumerate(cox_summary.iterrows()):
    ax.plot([row['CI_low'], row['CI_high']], [i, i],
            color=colors_hr[i], linewidth=2, alpha=0.7)

ax.axvline(x=1.0, color='black', linestyle='--', alpha=0.5, label='HR = 1 (no effect)')
ax.set_yticks(y_positions)
ax.set_yticklabels(cox_summary.index)
ax.set_xlabel('Hazard Ratio (HR) with 95% CI')
ax.set_title('Cox Model: Hazard Ratios for BCR\n(red = significant risk factor; green = protective; grey = non-significant)',
             fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)

legend_elements = [
    mpatches.Patch(color='#E84C4C', label='HR CI > 1 (risk factor)'),
    mpatches.Patch(color='#4CAF50', label='HR CI < 1 (protective)'),
    mpatches.Patch(color='#9E9E9E', label='CI crosses 1 (non-significant)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

### 🤔 Challenge Questions — Step 9

**9a.** In the forest plot, which features have confidence intervals that cross HR = 1.0? Should these features be retained in the final model? What is the risk of removing features that are non-significant in this cohort but clinically established (e.g., Gleason score)?

*Your answer here:*

---

**9b.** A patient's SHAP waterfall plot shows that their high NLR contributes +0.15 log-odds of BCR. The patient asks whether taking anti-inflammatory medication to lower their NLR would reduce their recurrence risk. How do you respond? What additional evidence (e.g., what study design) would be needed to establish a causal claim?

*Your answer here:*

---

**9c.** Compare the SHAP-based feature importance ranking (beeswarm) with the hazard ratio ranking (forest plot). Are they consistent? What explains any differences? Note that SHAP captures marginal contributions in the presence of correlated features, while Cox HRs are estimated jointly — correlated features like NLR and CALLY_Index may behave differently under the two frameworks.

*Your answer here:*

---

**9d.** The docx asks you to translate the inflammatory markers to clinical language. Write a 3-sentence patient-facing explanation of what a **high NLR** means biologically, why it is associated with BCR, and why this association does not mean that lowering NLR will prevent recurrence.

*Your answer here:*


---
# Summary: What You Built
---

| Step | Task | Key Output |
|------|------|------------|
| 0 | Setup & data dictionary | Structural vs. random missingness documented before any modeling |
| 1 | Data loading & duration | Survival duration for all 600 patients; BCR definitions by treatment type |
| 2 | EDA | Treatment-stratified missingness heatmap; PSA trajectory by BCR status; inflammatory marker distributions |
| 3 | Clinically-informed imputation | Treatment binary flags; within-group imputation only; leakage prevention |
| 4 | Feature engineering | PSADT, nadir depth, PSA velocity, CALLY verification, Gleason sum, ordinal stage |
| 5 | Train/test split | Stratified by BCR status AND treatment type; landmark leakage check |
| 6a | Logistic regression | Model A (clinical) vs. Model B (+ inflammatory) ablation; Risk_Grubu comparator; treatment-stratified AUROC |
| 6b | Mixed-effects model | Random intercept + slope for log-PSA trajectory; spaghetti plot with population overlay |
| 7 | Cox PH model | Kaplan-Meier by PSADT tertile; Cox fit with PH assumption check; individual survival curves |
| 8 | Evaluation | AUROC, AUPRC, C-index; treatment-stratified; calibration plot |
| 9 | Interpretation | SHAP beeswarm + waterfall; Cox hazard ratio forest plot; clinical translation of NLR/CALLY |

---

### Key Concepts Introduced in This Project

1. **Censoring** — a patient without BCR is not proven BCR-free; they were followed until `Son_Takip_Tarihi`
2. **Survival analysis (Cox PH)** — the principled model for time-to-event outcomes with censoring
3. **C-index** — AUROC generalized to censored data; probability that the predicted higher-risk patient recurs sooner
4. **Structural missingness** — missingness that encodes clinical reality, not data quality failure
5. **PSA doubling time (PSADT)** — log-linear PSA kinetics; short PSADT = aggressive recurrence
6. **CALLY Index** — composite inflammatory-nutritional marker: Albumin × Lenfosit / (NLR × 1000)
7. **Incremental predictive value** — Model A vs. B AUROC comparison quantifies the value of inflammatory markers
8. **Treatment-stratified evaluation** — pooled metrics can be misleadingly optimistic
9. **Correlation ≠ causation** — NLR and CALLY are correlates of systemic inflammation, not causal drivers of BCR
10. **Linear mixed-effects model** — separates fixed (population) and random (patient-specific) PSA trajectory effects
11. **Landmark analysis** — prevents temporal leakage by anchoring predictions to a fixed post-treatment timepoint
12. **Informative visit timing** — more frequent PSA measurements may themselves be a risk signal
13. **Spaghetti plot** — individual trajectory visualization with population-level model fit overlay
14. **Proportional hazards assumption** — must be verified; NLR or PSADT may have time-varying effects

---

### Alignment with Docx Project 12 Final Outcomes

| Docx Outcome | Implemented in |
|---|---|
| Mixed-effects trajectory model (random intercept + slope) | Step 6b |
| Response classification: Binary PSA50 ROC curve | Step 6a (AUROC) |
| Time-to-recurrence survival model with trajectory features | Step 7 |
| Spaghetti plot with population-level model fit overlay | Step 6b |

---

*I 320D Spring 2026 · University of Texas at Austin · School of Information*  
*Ammar Darkazanli, Ph.D., MBA · Lecturer*
